# 01.1 · NumPy: arrays, shapes and why vectors beat loops

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/01-working-with-data/01-numpy-arrays/lesson.ipynb)

Read the [README](README.md) first. Here you build the five-line example, check every number against your by-hand table, then scale to 541,910 real order lines and time it.

## 1. Five order lines as arrays

In [1]:
import numpy as np

qty = np.array([6, 6, 8, 6, 6])
price = np.array([2.55, 3.39, 2.75, 3.39, 3.39])

print("qty  :", qty,   "| shape", qty.shape,   "| dtype", qty.dtype,   "| ndim", qty.ndim,   "| size", qty.size)
print("price:", price, "| shape", price.shape, "| dtype", price.dtype)

qty  : [6 6 8 6 6] | shape (5,) | dtype int64 | ndim 1 | size 5
price: [2.55 3.39 2.75 3.39 3.39] | shape (5,) | dtype float64


`shape (5,)` means "five numbers in a row". `qty` got `int64` because every value was an integer; `price` got `float64` because of the decimals. One type per array.

## 2. Revenue: the loop way, then the array way

In [2]:
revenue_loop = []
for q, p in zip(qty.tolist(), price.tolist()):      # plain Python numbers, one pair at a time
    revenue_loop.append(round(q * p, 2))
print("loop  :", revenue_loop)

loop  : [15.3, 20.34, 22.0, 20.34, 20.34]


In [3]:
revenue = qty * price          # element-wise: one multiply instruction for the whole block
print("array :", revenue)
print("total :", revenue.sum().round(2), "  (by hand: 98.32)")
print("mean qty:", qty.mean(), "  (by hand: 6.4)")

array : [15.3  20.34 22.   20.34 20.34]
total : 98.32   (by hand: 98.32)
mean qty: 6.4   (by hand: 6.4)


Same numbers, no loop. `qty * price` is the **vectorised** form: NumPy runs the loop in C and hands you the finished array.

## 3. Shapes and `axis`

Stack quantity and price side by side to get a 5 × 2 grid, then sum down the columns and across the rows. Predict both results before you run the cell.

In [4]:
grid = np.column_stack([qty, price])
print(grid)
print("shape:", grid.shape, "| ndim:", grid.ndim, "| size:", grid.size)

[[6.   2.55]
 [6.   3.39]
 [8.   2.75]
 [6.   3.39]
 [6.   3.39]]
shape: (5, 2) | ndim: 2 | size: 10


In [5]:
print("sum(axis=0) → down each column   :", grid.sum(axis=0), " shape", grid.sum(axis=0).shape)
print("sum(axis=1) → across each row    :", grid.sum(axis=1), " shape", grid.sum(axis=1).shape)
print("sum()       → everything         :", grid.sum())

sum(axis=0) → down each column   : [32.   15.47]  shape (2,)
sum(axis=1) → across each row    : [ 8.55  9.39 10.75  9.39  9.39]  shape (5,)
sum()       → everything         : 47.47


`axis=0` removed the row axis and left one number per column (total quantity 32, total price 15.47). `axis=1` removed the column axis and left one number per row (quantity + price, meaningless here, but it shows the rule). **The axis you name is the one that disappears.**

## 4. Ways to create arrays you will actually use

In [6]:
print("zeros   :", np.zeros(4))
print("ones    :", np.ones((2, 3)))
print("arange  :", np.arange(0, 10, 2))          # start, stop (exclusive), step
print("linspace:", np.linspace(0, 1, 5))         # start, stop (inclusive), how many
print("full    :", np.full(3, 9.99))

zeros   : [0. 0. 0. 0.]
ones    : [[1. 1. 1.]
 [1. 1. 1.]]
arange  : [0 2 4 6 8]
linspace: [0.   0.25 0.5  0.75 1.  ]
full    : [9.99 9.99 9.99]


In [7]:
a = np.arange(12)
print(a)
print(a.reshape(3, 4))          # same 12 numbers, new shape
print(a.reshape(3, 4).shape)

[ 0  1  2  3  4  5  6  7  8  9 10 11]
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
(3, 4)


`reshape` does not copy or reorder the data; it changes how NumPy reads the same block. The product of the new shape must equal `size`.

## 5. The real data: 541,910 order lines

Load the 2010–2011 sheet (about a minute the first time), pull two columns into NumPy arrays, and time three ways of computing revenue.

In [8]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys, pandas as pd
sys.path.insert(0, str(data_path("download.py").parent))
from download import ensure_online_retail

retail = pd.read_excel(ensure_online_retail(), sheet_name="Year 2010-2011")
qty_real = retail["Quantity"].to_numpy()
price_real = retail["Price"].to_numpy()
print(qty_real.shape, qty_real.dtype, "|", price_real.shape, price_real.dtype)

(541910,) int64 | (541910,) float64


In [9]:
import time

t0 = time.perf_counter()
rev_loop = []
for q, p in zip(qty_real, price_real):
    rev_loop.append(q * p)
t_loop = time.perf_counter() - t0

t0 = time.perf_counter()
rev_comp = [q * p for q, p in zip(qty_real, price_real)]
t_comp = time.perf_counter() - t0

t0 = time.perf_counter()
rev_np = qty_real * price_real
t_np = time.perf_counter() - t0

print(f"for loop           : {t_loop*1000:8.1f} ms")
print(f"list comprehension : {t_comp*1000:8.1f} ms")
print(f"NumPy  qty * price : {t_np*1000:8.1f} ms   → {t_loop / t_np:.0f}× faster than the loop")

for loop           :     58.4 ms
list comprehension :     42.2 ms
NumPy  qty * price :      0.5 ms   → 123× faster than the loop


The exact speed-up depends on your machine, but it is **around a hundred times or more**, and the array line is also the shortest and clearest. Now the questions finance actually asked:

In [10]:
revenue_real = qty_real * price_real
print(f"total revenue (incl. returns as negatives): £{revenue_real.sum():,.2f}")
print(f"mean line revenue                         : £{revenue_real.mean():,.2f}")
print(f"largest single line                        : £{revenue_real.max():,.2f}")
print(f"smallest single line (a return)            : £{revenue_real.min():,.2f}")

total revenue (incl. returns as negatives): £9,747,765.93
mean line revenue                         : £17.99
largest single line                        : £168,469.60
smallest single line (a return)            : £-168,469.60


That largest line and that most negative line are the same transaction: a huge order that was placed and then cancelled. You will learn to spot and handle such rows in 01.5.

## 6. Why this matters for machine learning

In Module 02 a model will compute `prediction = m * x + c` for every row, measure the error, adjust `m` and `c`, and repeat thousands of times. With a Python loop over 500k rows, each round takes a tenth of a second; with arrays, a millisecond. The difference between a model that trains in a minute and one that trains overnight is entirely this lesson.

## Quiz answers

1. Every element has the same type and sits next to its neighbour in memory, so NumPy can run one compiled C loop with no per-element type checks.
2. Shape `(7, 3)`. `sum(axis=1)` adds across each row and returns shape `(7,)`: one total per row.
3. `float64`. NumPy picks the one type that can hold every value; a float can hold the integers, so everything becomes float.
4. Any three numbers whose product is 24, e.g. `(2, 3, 4)`, `(4, 6, 1)`, `(24, 1, 1)`.
5. Tiny data where clarity matters more than speed, or logic that differs per element in ways arrays cannot express (e.g. calling an external API per row).

**Next:** [01.2 Indexing, masks, broadcasting and random numbers](../02-numpy-indexing-broadcasting/)